# Double Pendulum Equations
To make the simulation, we need the equations of motion for the double pendulum. The system setup is a follows: we have a mass $m_1$ attached to the pendulum by a rod of length $l_1$ and a second mass $m_2$ attached to $m_1$ by a rod of length $l_2$.

<center>
    <img src="diagram.jpg" width="250"/>
</center>

We will use Lagrangian mechanics in order to find the equations of motion, along with the convention that gravity acts in the negative $y$ direction (this complicates things when it comes to rendering, but we can easily correct this at that stage). First, we have the position vectors of the two masses.
$$
    \mathbf{r}_1 = (l_1 \sin\theta_1, -l_1 \cos\theta_1)^\top, \mathbf{r}_2 = (l_1 \sin\theta_1 + l_2 \sin\theta_2, -l_1 \cos\theta_1 - l_2 \cos\theta_2)^\top
$$
where $\theta_1$ and $\theta_2$ are the angles as denoted on the diagram.

In [1]:
%display latex

In [2]:
var("l_1", "m_1", "l_2", "m_2", "t", "g")
theta_1_func = function("theta_1")
theta_2_func = function("theta_2")

theta_1 = theta_1_func(t)
theta_2 = theta_2_func(t)

In [3]:
x_1 = l_1 * sin(theta_1)
y_1 = -l_1 * cos(theta_1)
x_2 = x_1 + l_2 * sin(theta_2)
y_2 = y_1 - l_2 * cos(theta_2)

## Finding the Lagrangian
The first step is to derive the Lagrangian of the system
$$
    L = T - V
$$
where $T$ is the kinetic energy, and $V$ is the potential energy of the system.

### Kinetic Energy
The total kinetic energy for the system is $T = T_1 + T_2$ where
$$
    T_k = \frac{1}{2} m_k |\mathrm{\dot r}_k|^2
$$

In [4]:
# Calculate speed^2
x_1_dot = diff(x_1, t)
x_2_dot = diff(x_2, t)
y_1_dot = diff(y_1, t)
y_2_dot = diff(y_2, t)

speed_1 = x_1_dot^2 + y_1_dot^2
speed_2 = x_2_dot^2 + y_2_dot^2

kinetic_1 = (1/2) * m_1 * speed_1
kinetic_2 = (1/2) * m_2 * speed_2
kinetic = kinetic_1 + kinetic_2

### Potential Energy
The only potential energy in the simulation will be gravitational potential. Therefore, $V = V_1 + V_2$ where
$$
    V_k = -m_kgy_k
$$

In [5]:
potential_1 = -m_1 * g * y_1
potential_2 = -m_2 * g * y_2
potential = potential_1 + potential_2

In [6]:
lagrangian = (kinetic - potential).simplify_full()
lagrangian

1/2*l_2^2*m_2*diff(theta_2(t), t)^2 - g*l_2*m_2*cos(theta_2(t)) + 1/2*(l_1^2*m_1 + l_1^2*m_2)*diff(theta_1(t), t)^2 - (g*l_1*m_1 + g*l_1*m_2)*cos(theta_1(t)) + (l_1*l_2*m_2*cos(theta_1(t))*cos(theta_2(t))*diff(theta_1(t), t) + l_1*l_2*m_2*sin(theta_1(t))*sin(theta_2(t))*diff(theta_1(t), t))*diff(theta_2(t), t)

## Euler Lagrange Equations
The equations of motion then come from solving the Euler-Lagrange equations associated with the parameters of the model: $\theta_1$, $\theta_2$, i.e.
$$
    \frac{d}{dt}\left( \frac{\partial L}{\partial \dot\theta_k} \right) - \frac{\partial L}{\partial \theta_k} = 0
$$
for $k = 1, 2$.

SageMath's `diff` function only allows differentiation with respect to variables. Therefore, we must define new variables to stand-in for our $\dot\theta_k$ and $\ddot\theta_k$. Then, we can convert back for the final derivative wrt. $t$, so that chain rules can be handled appropriately.

In [7]:
theta_1_v = var("theta_1_v", latex_name=r"\theta_1")
theta_2_v = var("theta_2_v", latex_name=r"\theta_2")
theta_dot_1 = var("theta_dot_1", latex_name=r"\dot\theta_1")
theta_ddot_1 = var("theta_ddot_1", latex_name=r"\ddot\theta_1")
theta_dot_2 = var("theta_dot_2", latex_name=r"\dot\theta_2")
theta_ddot_2 = var("theta_ddot_2", latex_name=r"\ddot\theta_2")

to_vars = {
    theta_1: theta_1_v,
    theta_2: theta_2_v,
    diff(theta_1, t): theta_dot_1,
    diff(theta_2, t): theta_dot_2,
    diff(theta_1, t, 2): theta_ddot_1,
    diff(theta_2, t, 2): theta_ddot_2
}

to_func = {v: k for k, v in to_vars.items()}

lagrangian = lagrangian.subs(to_vars)
lagrangian

1/2*l_2^2*m_2*theta_dot_2^2 - g*l_2*m_2*cos(theta_2_v) + 1/2*(l_1^2*m_1 + l_1^2*m_2)*theta_dot_1^2 + (l_1*l_2*m_2*theta_dot_1*cos(theta_1_v)*cos(theta_2_v) + l_1*l_2*m_2*theta_dot_1*sin(theta_1_v)*sin(theta_2_v))*theta_dot_2 - (g*l_1*m_1 + g*l_1*m_2)*cos(theta_1_v)

In [8]:
to_vars

{theta_1(t): theta_1_v,
 theta_2(t): theta_2_v,
 diff(theta_1(t), t): theta_dot_1,
 diff(theta_2(t), t): theta_dot_2,
 diff(theta_1(t), t, t): theta_ddot_1,
 diff(theta_2(t), t, t): theta_ddot_2}

In [9]:
# k = 1
theta_deriv = diff(lagrangian, theta_1_v).subs(to_func)
theta_dot_deriv = diff(lagrangian, theta_dot_1).subs(to_func)

el_1 = (diff(theta_dot_deriv, t) - theta_deriv == 0).simplify_full()
el_1

(l_1*l_2*m_2*cos(theta_2(t))*sin(theta_1(t)) - l_1*l_2*m_2*cos(theta_1(t))*sin(theta_2(t)))*diff(theta_2(t), t)^2 - (g*l_1*m_1 + g*l_1*m_2)*sin(theta_1(t)) + (l_1^2*m_1 + l_1^2*m_2)*diff(theta_1(t), t, t) + (l_1*l_2*m_2*cos(theta_1(t))*cos(theta_2(t)) + l_1*l_2*m_2*sin(theta_1(t))*sin(theta_2(t)))*diff(theta_2(t), t, t) == 0

In [10]:
# k = 2
theta_deriv = diff(lagrangian, theta_2_v).subs(to_func)
theta_dot_deriv = diff(lagrangian, theta_dot_2).subs(to_func)

el_2 = (diff(theta_dot_deriv, t) - theta_deriv == 0).simplify_full()
el_2

l_2^2*m_2*diff(theta_2(t), t, t) - (l_1*l_2*m_2*sin(theta_1(t))*diff(theta_1(t), t)^2 - l_1*l_2*m_2*cos(theta_1(t))*diff(theta_1(t), t, t))*cos(theta_2(t)) + (l_1*l_2*m_2*cos(theta_1(t))*diff(theta_1(t), t)^2 + l_1*l_2*m_2*sin(theta_1(t))*diff(theta_1(t), t, t) - g*l_2*m_2)*sin(theta_2(t)) == 0

## Vectorising the ODEs
The Euler-Lagrange equations have given us two coupled ODEs. The easiest way to apply known numerical methods to this problem is to convert the system into a single first order vector ODE. The idea is as follows.

Suppose that we can write $\ddot\theta_1 = f_1(\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)$, $\ddot\theta_2 = f_2(\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)$. Then we can take a vector, $\mathbf{x} = (\theta_1, \theta_2, \dot\theta_1, \dot\theta_2)^\top$, then by the equations derived, we have that $\mathbf{\dot{x}} = (x_3, x_4, f_1(x_1, x_2, x_3, x_4), f_2(x_1, x_2, x_3, x_4))^\top$, a first order vector ODE.

The main, and only, step of interest at this stage of implementation is finding $f_1$ and $f_2$, which we can do quite easily using SageMath since the equations of motion are linear if $\ddot\theta_1$ and $\ddot\theta_2$. We will again need to convert back to variable notations, rather than using the functions directly in the expression.

In [11]:
el_1 = el_1.subs(to_vars)
el_2 = el_2.subs(to_vars)

In [12]:
theta_1_eq, theta_2_eq = solve([el_1, el_2], [theta_ddot_1, theta_ddot_2])[0]

In [13]:
theta_1_eq

theta_ddot_1 == (l_1*m_2*theta_dot_1^2*cos(theta_1_v)*cos(theta_2_v)^2*sin(theta_1_v) - (l_1*m_2*theta_dot_1^2*cos(theta_1_v) - g*m_2)*sin(theta_1_v)*sin(theta_2_v)^2 + (l_2*m_2*cos(theta_2_v)*sin(theta_1_v) - l_2*m_2*cos(theta_1_v)*sin(theta_2_v))*theta_dot_2^2 - (l_1*m_2*theta_dot_1^2*cos(theta_1_v)^2 - l_1*m_2*theta_dot_1^2*sin(theta_1_v)^2 - g*m_2*cos(theta_1_v))*cos(theta_2_v)*sin(theta_2_v) - (g*m_1 + g*m_2)*sin(theta_1_v))/(l_1*m_2*cos(theta_1_v)^2*cos(theta_2_v)^2 + 2*l_1*m_2*cos(theta_1_v)*cos(theta_2_v)*sin(theta_1_v)*sin(theta_2_v) + l_1*m_2*sin(theta_1_v)^2*sin(theta_2_v)^2 - l_1*m_1 - l_1*m_2)

In [14]:
theta_2_eq

theta_ddot_2 == -((l_2*m_2*cos(theta_1_v)*cos(theta_2_v)^2*sin(theta_1_v) - l_2*m_2*cos(theta_1_v)*sin(theta_1_v)*sin(theta_2_v)^2 - (l_2*m_2*cos(theta_1_v)^2 - l_2*m_2*sin(theta_1_v)^2)*cos(theta_2_v)*sin(theta_2_v))*theta_dot_2^2 + (l_1*m_1*theta_dot_1^2 + l_1*m_2*theta_dot_1^2 - (g*m_1 + g*m_2)*cos(theta_1_v))*cos(theta_2_v)*sin(theta_1_v) - ((g*m_1 + g*m_2)*sin(theta_1_v)^2 - g*m_1 - g*m_2 + (l_1*m_1*theta_dot_1^2 + l_1*m_2*theta_dot_1^2)*cos(theta_1_v))*sin(theta_2_v))/(l_2*m_2*cos(theta_1_v)^2*cos(theta_2_v)^2 + 2*l_2*m_2*cos(theta_1_v)*cos(theta_2_v)*sin(theta_1_v)*sin(theta_2_v) + l_2*m_2*sin(theta_1_v)^2*sin(theta_2_v)^2 - l_2*m_1 - l_2*m_2)

### Tidying Up
We've found the equations we need for implementation, however, they're quite long, and can probably be simplified and condensed down by quite a lot.

In [15]:
theta_1_eq = theta_1_eq.trig_reduce().expand().factor().simplify_rational()
theta_1_eq

theta_ddot_1 == -(2*l_2*m_2*theta_dot_2^2*sin(-theta_1_v + theta_2_v) + l_1*m_2*theta_dot_1^2*sin(-2*theta_1_v + 2*theta_2_v) - g*m_2*sin(-theta_1_v + 2*theta_2_v) + (2*g*m_1 + g*m_2)*sin(theta_1_v))/(l_1*m_2*cos(-2*theta_1_v + 2*theta_2_v) - 2*l_1*m_1 - l_1*m_2)

In [16]:
theta_2_eq = theta_2_eq.trig_reduce().expand().factor().simplify_rational()
theta_2_eq

theta_ddot_2 == (l_2*m_2*theta_dot_2^2*sin(-2*theta_1_v + 2*theta_2_v) + 2*(l_1*m_1 + l_1*m_2)*theta_dot_1^2*sin(-theta_1_v + theta_2_v) - (g*m_1 + g*m_2)*sin(-2*theta_1_v + theta_2_v) - (g*m_1 + g*m_2)*sin(theta_2_v))/(l_2*m_2*cos(-2*theta_1_v + 2*theta_2_v) - 2*l_2*m_1 - l_2*m_2)